# 🚁 DIP Project — Gün 2: SAHI Testi + Mosaic Augmentation
**Gün 1 notebook'u tamamlandıktan sonra bu notebook'u çalıştırın.**

## 0️⃣ NumPy Pin + Paket Kurulumu (Runtime başında çalıştır)

In [ ]:
import subprocess, sys

# NumPy<2 ve uyumlu cv2 — her zaman ilk kur
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'numpy<2',
    'opencv-python-headless==4.8.1.78',
])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'ultralytics==8.2.0',
    'sahi==0.11.15',
    'albumentations>=1.3.1,<2.0',
    'PyYAML',
    'tqdm',
])

import importlib; importlib.invalidate_caches()
import numpy as np
import cv2
print(f'✅ numpy={np.__version__}  cv2={cv2.__version__}')
print('\n✅ Paketler hazır, devam edebilirsin.')

## 1️⃣ Tüm Importlar

In [ ]:
import torch, os, time, cv2, yaml
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction, get_prediction
from sahi.slicing import slice_image
import albumentations as A

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PROJECT_DIR = '/content/drive/MyDrive/DIP_Project'
DATASET_DIR = f'{PROJECT_DIR}/datasets/VisDrone'
print(f'✅ Device: {DEVICE}')

## 2️⃣ Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(PROJECT_DIR, exist_ok=True)
print('✅ Drive bağlandı.')

## 3️⃣ SAHI — YOLOv8 Model Bağlantısı

In [ ]:
detection_model = AutoDetectionModel.from_pretrained(
    model_type='ultralytics',
    model_path='yolov8n.pt',
    confidence_threshold=0.3,
    device=DEVICE,
)
print('✅ SAHI AutoDetectionModel hazır!')

## 4️⃣ Test Görüntüsü Seç

In [ ]:
import urllib.request

IMG_PATH = '/content/test_aerial.jpg'
visdrone_imgs = list(Path(DATASET_DIR).rglob('*.jpg')) if os.path.exists(DATASET_DIR) else []

if visdrone_imgs:
    IMG_PATH = str(visdrone_imgs[0])
    print(f'✅ VisDrone görüntüsü: {IMG_PATH}')
else:
    urllib.request.urlretrieve('https://ultralytics.com/images/zidane.jpg', IMG_PATH)
    print('✅ Fallback görüntü indirildi.')

img = cv2.imread(IMG_PATH)
h, w = img.shape[:2]
print(f'Görüntü: {w}x{h} px')

## 5️⃣ SAHI Parametre Karşılaştırması
640×640 %20 overlap — 512×512 %30 overlap — 320×320 %20 — No Slice

In [ ]:
configs = [
    {'name': 'No Slice (Baseline)', 'slice': False, 'size': None, 'overlap': None},
    {'name': '640×640 / %20 Overlap', 'slice': True, 'size': 640, 'overlap': 0.2},
    {'name': '512×512 / %30 Overlap', 'slice': True, 'size': 512, 'overlap': 0.3},
    {'name': '320×320 / %20 Overlap', 'slice': True, 'size': 320, 'overlap': 0.2},
]

results_summary = []
for cfg in configs:
    t0 = time.time()
    if cfg['slice']:
        res = get_sliced_prediction(
            IMG_PATH, detection_model,
            slice_height=cfg['size'], slice_width=cfg['size'],
            overlap_height_ratio=cfg['overlap'], overlap_width_ratio=cfg['overlap'],
            verbose=0,
        )
    else:
        res = get_prediction(IMG_PATH, detection_model)
    elapsed = time.time() - t0
    n = len(res.object_prediction_list)
    results_summary.append({'Config': cfg['name'], 'Detections': n, 'Time (s)': round(elapsed, 2)})
    print(f"[{cfg['name']}]  Tespit: {n}  |  Süre: {elapsed:.2f}s")

print('\n✅ Karşılaştırma tamamlandı!')

In [ ]:
import pandas as pd
df = pd.DataFrame(results_summary)
print(df.to_string(index=False))

colors = ['#e74c3c','#3498db','#2ecc71','#f39c12']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(df['Config'], df['Detections'], color=colors)
axes[0].set_title('Tespit Sayısı'); axes[0].set_ylabel('Nesne Sayısı')
axes[0].set_xticklabels(df['Config'], rotation=15, ha='right', fontsize=8)
axes[1].bar(df['Config'], df['Time (s)'], color=colors)
axes[1].set_title('Inference Süresi (s)'); axes[1].set_ylabel('Saniye')
axes[1].set_xticklabels(df['Config'], rotation=15, ha='right', fontsize=8)
plt.suptitle('SAHI Parametre Karşılaştırması', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/sahi_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6️⃣ En İyi Config ile Annotated Çıktı (640×640 %20)

In [ ]:
best_result = get_sliced_prediction(
    IMG_PATH, detection_model,
    slice_height=640, slice_width=640,
    overlap_height_ratio=0.2, overlap_width_ratio=0.2,
    verbose=0,
)
best_result.export_visuals(
    export_dir=PROJECT_DIR,
    file_name='sahi_best_result',
    export_format='png',
)
out_img = plt.imread(f'{PROJECT_DIR}/sahi_best_result.png')
plt.figure(figsize=(14, 8))
plt.imshow(out_img); plt.axis('off')
plt.title(f'SAHI 640×640 %20 Overlap — {len(best_result.object_prediction_list)} tespit', fontsize=13)
plt.show()

## 7️⃣ Slice Grid Görselleştirmesi

In [ ]:
slice_result = slice_image(
    image=IMG_PATH,
    output_file_name='slice',
    output_dir='/content/slices/',
    slice_height=640, slice_width=640,
    overlap_height_ratio=0.2, overlap_width_ratio=0.2,
)
n_slices = len(slice_result.sliced_image_list)
print(f'Toplam patch sayısı: {n_slices}')

cols = 3
rows = min(3, (n_slices + cols - 1) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows))
for i, ax in enumerate(np.array(axes).flat):
    if i < n_slices:
        ax.imshow(slice_result.sliced_image_list[i].image)
        ax.set_title(f'Patch {i+1}', fontsize=8)
    ax.axis('off')
plt.suptitle(f'Slice Grid ({n_slices} patch, 640×640, %20 overlap)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/slice_grid.png', dpi=150)
plt.show()

## 8️⃣ VisDrone → YOLO Format Dönüşümü

In [ ]:
VISDRONE_TO_YOLO = {1:0, 2:1, 3:2, 4:3, 5:4, 6:5, 7:6, 8:7, 9:8, 10:9}
CLASS_NAMES = ['pedestrian','people','bicycle','car','van','truck',
               'tricycle','awning-tricycle','bus','motor']

def convert_visdrone_to_yolo(ann_path, img_w, img_h):
    lines = []
    with open(ann_path) as f:
        for line in f:
            p = line.strip().split(',')
            if len(p) < 6: continue
            x, y, w, h, cat = int(p[0]), int(p[1]), int(p[2]), int(p[3]), int(p[5])
            if cat not in VISDRONE_TO_YOLO or w == 0 or h == 0: continue
            cx = min(max((x + w/2) / img_w, 0), 1)
            cy = min(max((y + h/2) / img_h, 0), 1)
            nw = min(w / img_w, 1); nh = min(h / img_h, 1)
            lines.append(f'{VISDRONE_TO_YOLO[cat]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
    return lines

def batch_convert(split='train'):
    import shutil
    ann_dir = Path(DATASET_DIR) / f'VisDrone2019-DET-{split}' / 'annotations'
    img_dir = Path(DATASET_DIR) / f'VisDrone2019-DET-{split}' / 'images'
    lbl_out = Path(DATASET_DIR) / 'labels' / split
    img_out = Path(DATASET_DIR) / 'images' / split
    lbl_out.mkdir(parents=True, exist_ok=True)
    img_out.mkdir(parents=True, exist_ok=True)
    if not ann_dir.exists():
        print(f'⚠️ {split} bulunamadı: {ann_dir}'); return
    anns = list(ann_dir.glob('*.txt'))
    print(f'[{split}] {len(anns)} annotation dönüştürülüyor...')
    for ann in anns:
        img_p = img_dir / (ann.stem + '.jpg')
        if not img_p.exists(): continue
        img = cv2.imread(str(img_p))
        if img is None: continue
        h2, w2 = img.shape[:2]
        ylines = convert_visdrone_to_yolo(ann, w2, h2)
        (lbl_out / (ann.stem + '.txt')).write_text('\n'.join(ylines))
        shutil.copy2(img_p, img_out / img_p.name)
    print(f'  ✅ {split} → {lbl_out}')

for split in ['train', 'val', 'test-dev']:
    batch_convert(split)
print('\n✅ Tüm dönüşümler tamamlandı!')

## 9️⃣ data.yaml Oluştur

In [ ]:
import yaml

data_config = {
    'path': DATASET_DIR,
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test-dev',
    'nc': 10,
    'names': CLASS_NAMES,
}
yaml_path = f'{PROJECT_DIR}/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False, allow_unicode=True)

print('✅ data.yaml oluşturuldu:')
print(open(yaml_path).read())

## 🔟 Mosaic Augmentation Pipeline

In [ ]:
import albumentations as A

augmentation_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=40, val_shift_limit=30, p=0.5),
    A.GaussNoise(var_limit=(10, 50), p=0.2),
    A.MotionBlur(blur_limit=5, p=0.1),
    A.Perspective(scale=(0.02, 0.05), p=0.1),
    A.Resize(640, 640),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'],
                            min_area=100, min_visibility=0.3))

print(f'✅ Augmentation pipeline hazır! ({len(augmentation_pipeline.transforms)} transform)')

In [ ]:
def mosaic_4images(img_paths, img_size=640):
    s = img_size
    canvas = np.zeros((s*2, s*2, 3), dtype=np.uint8)
    positions = [(0,0),(s,0),(0,s),(s,s)]
    for (x, y), ip in zip(positions, img_paths):
        img = cv2.imread(ip)
        if img is None:
            img = np.random.randint(50, 200, (s, s, 3), dtype=np.uint8)
        canvas[y:y+s, x:x+s] = cv2.resize(img, (s, s))
    cv2.line(canvas, (s,0), (s,2*s), (255,255,0), 2)
    cv2.line(canvas, (0,s), (2*s,s), (255,255,0), 2)
    return canvas

img_list = list(Path(DATASET_DIR).rglob('*.jpg'))[:4] if os.path.exists(DATASET_DIR) else []
if len(img_list) == 0: img_list = [IMG_PATH] * 4
elif len(img_list) < 4: img_list = (img_list * 4)[:4]

mosaic = mosaic_4images([str(p) for p in img_list])
plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(mosaic, cv2.COLOR_BGR2RGB))
plt.title('Mosaic Augmentation (4 görüntü)', fontsize=13, fontweight='bold')
plt.axis('off')
plt.savefig(f'{PROJECT_DIR}/mosaic_example.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Mosaic kaydedildi.')

In [ ]:
# Augmentation grid: orijinal + 7 augmented örnek
sample = cv2.cvtColor(cv2.imread(IMG_PATH), cv2.COLOR_BGR2RGB)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes[0,0].imshow(sample); axes[0,0].set_title('Orijinal', fontweight='bold'); axes[0,0].axis('off')
for i in range(1, 8):
    aug = augmentation_pipeline(image=sample, bboxes=[], class_labels=[])
    r, c = divmod(i, 4)
    axes[r,c].imshow(aug['image']); axes[r,c].set_title(f'Aug #{i}', fontsize=9); axes[r,c].axis('off')
plt.suptitle('Albumentations Augmentation Örnekleri', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/augmentation_grid.png', dpi=150)
plt.show()
print('\n✅ GÜN 2 TAMAMLANDI!')

---
## ✅ Gün 2 Özet

| Adım | Durum |
|------|-------|
| NumPy<2 + cv2 uyum fix | ✅ |
| SAHI AutoDetectionModel | ✅ |
| Parametre karşılaştırması (4 config) | ✅ |
| Annotated çıktı (640×640 %20) | ✅ |
| Slice grid (9 patch) | ✅ |
| VisDrone → YOLO dönüşüm | ✅ |
| data.yaml | ✅ |
| Mosaic augmentation | ✅ |
| Augmentation grid | ✅ |

**Sonraki (Gün 3):** `yolo detect train data=data.yaml model=yolov8s.pt imgsz=640 epochs=50`